In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from pytorch3d.vis.plotly_vis import plot_scene
from data_tools import adv_dataset, adversarial_patch_3d

from attack_utils import *

CKPT_PATH = "output/train/full_attack_pointpillar/final_adversarial_patch_checkpoint.pt"

In [6]:
state_dict = torch.load(CKPT_PATH)

In [7]:
universal_adv_patch_car = single_sphere(scale=adv_dataset.CAR_ADV_PATCH_SCALE)
universal_adv_patch_car.load_parameter(state_dict["universal_adv_patch_car"])

print(state_dict["universal_adv_patch_car"])

mesh vertex count : torch.Size([162, 3])
[tensor([-0.3321,  0.2174,  0.0000], device='cuda:0', requires_grad=True), tensor([-0.4214], device='cuda:0', requires_grad=True), tensor([[-0.2404, -0.1145,  0.0000],
        [ 0.0618,  0.0256,  0.0000],
        [-0.1969, -0.1515,  0.0000],
        [-1.2392, -1.1600,  0.0000],
        [ 0.0000,  0.3925,  0.0000],
        [ 0.0000,  0.5269,  0.0000],
        [ 0.0000, -0.5382,  0.0000],
        [ 0.0000, -1.5368,  0.0000],
        [-0.3467,  0.0000,  0.0000],
        [-0.6076,  0.0000,  0.0000],
        [ 0.1969,  0.0000,  0.0000],
        [-0.3281,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000],
        [-0.6797, -0.5507,  0.0000],
        [-0.2159, -0.4569,  0.0000],
        [ 0.4337,  0.4886,  0.0000],
        [ 0.2275,  0.2574,  0.0000],
        [-0.4505, -0.2600,  0.0000],
        [-0.8068, -0.6257,  0.0000],
        [-0.4333, -0.4515,  0.0000],
        [ 0.1459,  0.0537,  0.0000],
        [ 0.0000,  0.0000,  0.0000],
        [ 0.44

In [8]:
fig = plot_scene({
                    "original": {
                        "mesh_1": universal_adv_patch_car.get_basic_meshes()
                    },
                    "adversarial": {
                        "mesh_1": universal_adv_patch_car.get_deformed_meshes()
                    },
                }, ncols=2)
fig.update_layout(height=400, width=800)
fig.show()

In [3]:
from data_tools import simple_cubic_meshes, join_meshes_as_batch
from pytorch3d.vis.plotly_vis import plot_scene

test = simple_cubic_meshes(cubic_level=2)
meshes_batch = join_meshes_as_batch(test.get_deformed_lattice())

In [4]:
fig = plot_scene({
                    "original": {
                        "mesh_1": meshes_batch
                    }
                }, ncols=1)
fig.update_layout(height=400, width=400)
fig.show()

In [6]:
import pandas as pd

def parse_table(data):
    # 将数据转换为DataFrame
    lines = data.strip().split('\n')
    rows = [line.split() for line in lines]
    columns = ["Category", "Value1", "Value2", "Value3", "Value4"]

    df = pd.DataFrame(rows, columns=columns)

    # 将数值列转换为浮点数
    df[["Value1", "Value2", "Value3", "Value4"]] = df[["Value1", "Value2", "Value3", "Value4"]].astype(float)

    print(df)

    clean_values = df.loc[df['Category'] == 'Clean', ["Value1", "Value2", "Value3", "Value4"]].values[0]

    # 计算每个类别相对于Clean的百分比
    percentage_df = df.copy()
    percentage_df[["Value1", "Value2", "Value3", "Value4"]] = 100 - df[["Value1", "Value2", "Value3", "Value4"]].div(clean_values) * 100

    print("Original DataFrame:")
    print(df)
    print("\nPercentage DataFrame:")
    print(percentage_df)

In [9]:
data_1 = """
Clean       86.7564 85.6744 88.9160 89.1513
Vanilla     80.6136 76.9694 83.1086 88.1049
FullAttack  79.4522 74.2181 86.4966 77.4980
IouFrozen   78.6085 72.7279 86.9074 76.9161
LogitFrozen 76.5190 74.8475 85.5587 76.4687
"""

data_2 = """
Clean	77.5743	78.6668	84.3717	84.4291
Vanilla	57.1795	66.5986	66.2274	72.8757
FullAttack	55.8176	63.2555	78.1210	65.9220
IouFrozen	56.8577	61.7680	78.5877	65.7358
LogitFrozen	48.4185	64.2011	77.2747	66.1394
"""


parse_table(data_1)

parse_table(data_2)

      Category   Value1   Value2   Value3   Value4
0        Clean  86.7564  85.6744  88.9160  89.1513
1      Vanilla  80.6136  76.9694  83.1086  88.1049
2   FullAttack  79.4522  74.2181  86.4966  77.4980
3    IouFrozen  78.6085  72.7279  86.9074  76.9161
4  LogitFrozen  76.5190  74.8475  85.5587  76.4687
Original DataFrame:
      Category   Value1   Value2   Value3   Value4
0        Clean  86.7564  85.6744  88.9160  89.1513
1      Vanilla  80.6136  76.9694  83.1086  88.1049
2   FullAttack  79.4522  74.2181  86.4966  77.4980
3    IouFrozen  78.6085  72.7279  86.9074  76.9161
4  LogitFrozen  76.5190  74.8475  85.5587  76.4687

Percentage DataFrame:
      Category     Value1     Value2    Value3     Value4
0        Clean   0.000000   0.000000  0.000000   0.000000
1      Vanilla   7.080515  10.160561  6.531333   1.173735
2   FullAttack   8.419206  13.371906  2.720995  13.071374
3    IouFrozen   9.391699  15.111282  2.258986  13.724085
4  LogitFrozen  11.800167  12.637264  3.775811  14.2259